<h4> Install / imports </h4>

In [ ]:
import random
import numpy as np
from sklearn.model_selection import train_test_split
from datasets import load_dataset
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from seqeval.metrics import classification_report


<h4> Extracting Sentences and Tags </h4>

In [5]:
dataset = load_dataset("lhoestq/conll2003")

# Split dataset into train, validation, and test sets
train_data = dataset["train"]
val_data   = dataset["validation"]
test_data  = dataset["test"]

# Extract sentences (list of tokens per sentence)
sentences_train = train_data["tokens"]
sentences_val = val_data["tokens"]
sentences_test = test_data["tokens"]

# Extract tags (list of tag IDs per sentence)
tags_train = train_data["ner_tags"]
tags_val  = val_data["ner_tags"]
tags_test = test_data["ner_tags"]

# Number of unique tags
all_tags = np.concatenate(tags_train)   # flatten into one long array
num_tags = len(np.unique(all_tags))

print("Number of distinct tags:", num_tags)

Number of distinct tags: 9


In [6]:
print(sentences_train[0])
print(tags_train[0])

['EU', 'rejects', 'German', 'call', 'to', 'boycott', 'British', 'lamb', '.']
[3, 0, 7, 0, 0, 0, 7, 0, 0]


<h4> Split the data by sentence (train / val / test) </h4>

In [7]:
def split_dataset(sentences, tags, test_size=0.15, val_size=0.15, random_state=42):
    # first split out test
    s_temp, s_test, t_temp, t_test = train_test_split(
        sentences, tags, test_size=test_size, random_state=random_state
    )
    # then split remaining into train + val (val_size relative to original -> adjust)
    val_ratio_of_temp = val_size / (1 - test_size)
    s_train, s_val, t_train, t_val = train_test_split(
        s_temp, t_temp, test_size=val_ratio_of_temp, random_state=random_state
    )
    return (s_train, t_train), (s_val, t_val), (s_test, t_test)

<h4> Convert tokens → embeddings, and flatten (FFNN uses each token independently) </h4>

Preprocessing of the embeddings_index

In [8]:
embedding_matrix = torch.load("word2vec_embeddings.pt", map_location="cpu")
vocab = torch.load("vocab_list.pt", map_location="cpu")
embeddings = {}
for token, idx in vocab.items():    # works only if vocab is dict token→index
    embeddings[token] = embedding_matrix[idx]

embedding_dim = embedding_matrix.shape[1]


Convert tokens → embeddings, and flatten

In [9]:
def tokens_to_flat_embeddings(sentences, tags, embeddings_index, embedding_dim, oov_strategy="zero"):
    """
    Convert list-of-sentences and list-of-tag-lists to flattened arrays:
     - X: (N_tokens, embedding_dim) numpy array
     - y: (N_tokens,) numpy array of ints

    oov_strategy: "zero" (use 0 vector) or "random" (random normal)
    """
    X_list = []
    y_list = []
    for sent, tag_seq in zip(sentences, tags):
        for tok, lab in zip(sent, tag_seq):
            if tok in embeddings_index:
                vec = embeddings_index[tok]
            else:
                if oov_strategy == "zero":
                    vec = np.zeros(embedding_dim, dtype=np.float32)
                else:
                    vec = np.random.normal(scale=0.1, size=(embedding_dim,)).astype(np.float32)
                    
            if isinstance(vec, torch.Tensor):
                vec = vec.detach().cpu().numpy()
            
            # Ensure float32 dtype
            vec = vec.astype(np.float32)
            
            X_list.append(vec)
            y_list.append(int(lab))
    X = np.vstack(X_list) if len(X_list) > 0 else np.zeros((0, embedding_dim), dtype=np.float32)
    y = np.array(y_list, dtype=np.int64)
    return X, y

<h4> Build DataLoaders </h4>

In [10]:
def make_dataloader(X, y, batch_size=128, shuffle=True):
    X_t = torch.from_numpy(X)           # shape (N, embedding_dim), dtype float32
    y_t = torch.from_numpy(y).long()    # shape (N,), dtype int64
    ds = TensorDataset(X_t, y_t)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)

<h4> Implement the Feed-Forward model (PyTorch) </h4>

In [11]:
class FFNN_NER(nn.Module):
    def __init__(self, embedding_dim, hidden_dim, num_tags, dropout=0.2):
        super(FFNN_NER, self).__init__()
        self.fc1 = nn.Linear(embedding_dim, hidden_dim)
        self.act = nn.ReLU()
        self.drop = nn.Dropout(dropout)
        self.fc2 = nn.Linear(hidden_dim, num_tags)
    
    def forward(self, x):
        # x: (batch, embedding_dim)
        h = self.act(self.fc1(x))
        h = self.drop(h)
        logits = self.fc2(h)   # (batch, num_tags)
        return logits

<h4> Training loop (train + validate each epoch) </h4>

In [12]:
def train_model(model, train_loader, val_loader, num_epochs=10, lr=1e-3, device='cpu'):
    model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    best_val_loss = float('inf')
    best_state = None

    for epoch in range(1, num_epochs+1):
        # --- training ---
        model.train()
        total_loss = 0.0
        total_tokens = 0
        for X_batch, y_batch in train_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            optimizer.zero_grad()
            logits = model(X_batch)               # (B, num_tags)
            loss = criterion(logits, y_batch)     # averaged over batch
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * X_batch.size(0)
            total_tokens += X_batch.size(0)
        train_loss = total_loss / total_tokens

        # --- validation ---
        model.eval()
        val_loss = 0.0
        val_tokens = 0
        correct = 0
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch = X_batch.to(device)
                y_batch = y_batch.to(device)
                logits = model(X_batch)
                loss = criterion(logits, y_batch)
                val_loss += loss.item() * X_batch.size(0)
                val_tokens += X_batch.size(0)
                preds = logits.argmax(dim=1)
                correct += (preds == y_batch).sum().item()
        val_loss = val_loss / val_tokens
        val_acc = correct / val_tokens

        print(f"Epoch {epoch:02d} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | val_acc={val_acc:.4f}")

        # save best
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = model.state_dict()

    # load best state
    if best_state is not None:
        model.load_state_dict(best_state)
    return model

<h4> Test evaluation </h4>

In [ ]:
from sklearn.metrics import classification_report
import torch.nn.functional as F

def evaluate_model(model, test_loader, idx_to_tag, device='cpu'):
    model.to(device)
    model.eval()
    criterion = nn.CrossEntropyLoss(reduction='sum')

    total_loss = 0.0
    total_tokens = 0
    correct_tokens = 0

    all_true_tags = []
    all_pred_tags = []

    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            logits = model(X_batch) 

            logits = logits.view(-1, logits.shape[-1])     # (batch*seq_len, num_tags)
            y_batch = y_batch.view(-1)                     # (batch*seq_len)

            loss = criterion(logits, y_batch)
            total_loss += loss.item()

            preds = torch.argmax(logits, dim=-1)
            correct_tokens += (preds == y_batch).sum().item()
            total_tokens += y_batch.numel()

            for t, p in zip(y_batch.tolist(), preds.tolist()):
                all_true_tags.append(idx_to_tag[t])
                all_pred_tags.append(idx_to_tag[p])

    avg_loss = total_loss / total_tokens
    accuracy = correct_tokens / total_tokens

    print(f"Test loss: {avg_loss:.4f} | Token-level accuracy: {accuracy:.4f}")
    print("\nDetailed classification report:")
    print(classification_report(all_true_tags, all_pred_tags, digits=4))


<h4> Usages </h4>

Data splitting

In [14]:
(train_sents, train_tags), (val_sents, val_tags), (test_sents, test_tags) = split_dataset(sentences_train, tags_train)

Tokens to embeddings

In [15]:
X_train, y_train = tokens_to_flat_embeddings(train_sents, train_tags, embeddings, embedding_dim)
X_val, y_val = tokens_to_flat_embeddings(val_sents, val_tags, embeddings, embedding_dim)
X_test, y_test = tokens_to_flat_embeddings(test_sents, test_tags, embeddings, embedding_dim)

Build DataLoaders

In [16]:
train_loader = make_dataloader(X_train, y_train, batch_size=256, shuffle=True)
val_loader   = make_dataloader(X_val, y_val, batch_size=256, shuffle=False)
test_loader  = make_dataloader(X_test, y_test, batch_size=256, shuffle=False)

Building the Feed-Forward model

In [17]:
hidden_dim = 128
model = FFNN_NER(embedding_dim=embedding_dim, hidden_dim=hidden_dim, num_tags=num_tags, dropout=0.2)

Training the Model

In [18]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = train_model(model, train_loader, val_loader, num_epochs=10, lr=1e-3, device=device)

Epoch 01 | train_loss=0.6896 | val_loss=0.6061 | val_acc=0.8319
Epoch 02 | train_loss=0.6047 | val_loss=0.6033 | val_acc=0.8323
Epoch 03 | train_loss=0.6025 | val_loss=0.6032 | val_acc=0.8323
Epoch 04 | train_loss=0.6011 | val_loss=0.6032 | val_acc=0.8323
Epoch 05 | train_loss=0.6004 | val_loss=0.6035 | val_acc=0.8322
Epoch 06 | train_loss=0.6006 | val_loss=0.6038 | val_acc=0.8322
Epoch 07 | train_loss=0.5999 | val_loss=0.6035 | val_acc=0.8323
Epoch 08 | train_loss=0.5997 | val_loss=0.6040 | val_acc=0.8324
Epoch 09 | train_loss=0.5995 | val_loss=0.6039 | val_acc=0.8323
Epoch 10 | train_loss=0.5995 | val_loss=0.6046 | val_acc=0.8324


Testing the Model

In [37]:
idx_to_tag = {
    0: "O",
    1: "B-PER",
    2: "I-PER",
    3: "B-ORG",
    4: "I-ORG",
    5: "B-LOC",
    6: "I-LOC",
    7: "B-MISC",
    8: "I-MISC"
}

In [44]:
evaluate_model(model, test_loader, idx_to_tag)

Test loss: 0.5903 | Token-level accuracy: 0.8384

Detailed classification report:
              precision    recall  f1-score   support

       B-LOC     0.0000    0.0000    0.0000      1036
      B-MISC     0.5000    0.0020    0.0039       512
       B-ORG     0.5000    0.0011    0.0022       908
       B-PER     0.0000    0.0000    0.0000       954
       I-LOC     0.0000    0.0000    0.0000       202
      I-MISC     0.7500    0.0339    0.0649       177
       I-ORG     1.0000    0.0018    0.0037       542
       I-PER     0.5000    0.0061    0.0121       654
           O     0.8385    0.9999    0.9121     25799

    accuracy                         0.8384     30784
   macro avg     0.4543    0.1161    0.1110     30784
weighted avg     0.7584    0.8384    0.7653     30784



c:\Users\Lenovo\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Lenovo\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Lenovo\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
torch.save(model.state_dict(), "ffnn_ner_best.pt")

Testing a real example!!

In [33]:
def predict_tags_for_sentence(model, sentence, embeddings_index, embedding_dim, idx_to_tag):
    
    # Convert tokens to embeddings (like test data)
    X, _ = tokens_to_flat_embeddings([sentence], [[0]*len(sentence)], 
                                     embeddings_index, embedding_dim)
    
    # Convert X to torch
    X_t = torch.from_numpy(X).float().unsqueeze(0)   # shape (1, N_tokens, embedding_dim)

    # Model forward
    model.eval()
    with torch.no_grad():
        logits = model(X_t)          # shape (1, N_tokens, num_tags)
        print(logits)
        preds = torch.argmax(logits, dim=-1).squeeze(0)  # shape (N_tokens,)
    
    # Convert tag indices → tag labels
    print(preds)
    predicted_tags = [idx_to_tag[int(p)] for p in preds]
    return predicted_tags

sentence = ['EU', 'rejects', 'German', 'call', 'to', 'boycott', 'British', 'lamb', '.']
sentence = [tok.lower() for tok in sentence]  # if training used lowercase

predicted = predict_tags_for_sentence(model, sentence, embeddings, embedding_dim, idx_to_tag)

for tok, tag in zip(sentence, predicted):
    print(tok, "→", tag)


tensor([[[ 12.9607, -11.2861, -10.4171, -11.5175,  -5.1874, -12.5095, -16.1795,
           -9.7824,  -8.3283],
         [  5.8175,  -5.8696,  -3.7851,  -4.2449,  -5.0364,  -7.2380,  -8.6050,
           -6.4000,  -9.0875],
         [  3.9111,  -3.2287,  -2.9002,  -2.9040,  -3.0320,  -3.8407,  -3.9764,
           -3.1239,  -4.8662],
         [  9.4587, -10.0643,  -9.8466,  -8.4865,  -5.6332, -10.9195,  -8.9857,
           -8.0574,  -1.4542],
         [  6.1704,  -5.1021,  -5.4481,  -4.8366,  -2.5887,  -5.3549,  -6.3622,
           -5.6003,  -3.5502],
         [  6.3111,  -7.4400,  -5.8684,  -7.1586,  -1.0931,  -8.4392,  -6.5575,
           -5.8244,  -4.1468],
         [  6.7742,  -7.3040,  -6.6219,  -5.4849,  -4.7622,  -7.6666,  -6.1069,
           -6.2261,  -2.0300],
         [ 10.4402, -11.9973,  -8.9171, -11.2770,  -5.8218, -14.5674, -11.8066,
          -11.1193,  -4.9528],
         [  1.3808,  -0.5400,  -0.8595,  -0.5396,  -1.1024,  -0.5248,  -2.3503,
           -1.2172,  -2.3277]]])

In [30]:
print(sum(p.abs().sum().item() for p in model.parameters()))

2392.265491127968
